# Agentic BL Backtest — Inspection Notebook

Notebook version of `run_example.py`: runs the mocked `AgenticViewWeighTarget` strategy against an equal-weight baseline on real prices from `price_data.db`, then plots the same things the frontend shows (equity curve, per-asset weight allocation, prior-vs-posterior-style weight comparison, and a performance metrics grid) using Plotly for quick visual inspection.

In [1]:
import sys
from pathlib import Path


def _find_backend_dir(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "app" / "services" / "bl_backtest").is_dir():
            return p
    raise RuntimeError("Could not locate backend/ directory from " + str(start))


_BACKEND_DIR = _find_backend_dir(Path.cwd())
if str(_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(_BACKEND_DIR))

import bt
import pandas as pd
import plotly.graph_objects as go

from app.services.bl_backtest.run_example import (
    HYPOTHESIS,
    _NON_EQUITY_ASSETS,
    build_agentic_strategy,
    build_equal_weight_strategy,
)
from app.services.price_data.load_data import load_market_data

# Palette matches the app's chart colours (BacktestPage / BLAllocationChart).
COLOUR_AGENTIC = "#60a5fa"   # blue — primary / posterior
COLOUR_EQUAL = "#f59e0b"     # amber — baseline / prior
ASSET_COLOURS = ['#60a5fa', '#34d399', '#f59e0b', '#f87171', '#a78bfa', '#38bdf8', '#fb923c', '#4ade80']

## Run the backtest

Same wiring as `run_example.py`: equities-only universe, monthly rebalance, mocked agent weights vs. equal weighting.

In [2]:
price_df, _market_caps, _B, _factor_names, all_assets = load_market_data()
universe = [a for a in all_assets if a not in _NON_EQUITY_ASSETS]

agentic_bt = bt.Backtest(build_agentic_strategy(universe, HYPOTHESIS), price_df[universe])
equal_weight_bt = bt.Backtest(build_equal_weight_strategy(universe), price_df[universe])

result = bt.run(agentic_bt, equal_weight_bt)
result.stats


Reading from SQLite database: None
✓ Successfully read 1254 rows from table 'price_history'

✓ Loaded real price data:
  Assets: 13 (AAPL, AMZN, BAC, BND, GLD, GOOGL, JNJ, JPM, MSFT, PG, TSLA, VNQ, WMT)
  Date range: 2021-03-19 00:00:00 to 2026-03-17 00:00:00
  Trading days: 1254


,AgenticBL,EqualWeight
start,2021-03-18 00:00:00,2021-03-18 00:00:00
end,2026-03-17 00:00:00,2026-03-17 00:00:00
rf,0.0,0.0
total_return,1.340297,1.224431
cagr,0.185509,0.173522
max_drawdown,-0.263019,-0.283057
calmar,0.705304,0.613029
mtd,-0.02567,-0.025648
three_month,-0.022937,-0.03062
six_month,0.047346,0.03403


## Equity curve

Equivalent of the app's **Equity Curve** chart (Portfolio mode) — indexed value (base = 100) for each strategy over time.

In [3]:
prices = result.prices

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=prices.index, y=prices["AgenticBL"],
    name="AgenticBL", mode="lines",
    line=dict(color=COLOUR_AGENTIC, width=2),
))
fig.add_trace(go.Scatter(
    x=prices.index, y=prices["EqualWeight"],
    name="EqualWeight", mode="lines",
    line=dict(color=COLOUR_EQUAL, width=2),
))
fig.update_layout(
    title="Equity Curve (indexed, base=100)",
    xaxis_title="Date", yaxis_title="Indexed Value",
    template="plotly_dark", height=420,
)
fig.show()

## Performance metrics

Equivalent of the app's **Performance Metrics** grid (`BacktestPage` metric cards).

In [5]:
stats = result.stats
metric_rows = [
    ("Total Return", "total_return", 100),
    ("Ann. Return (CAGR)", "cagr", 100),
    ("Volatility (ann.)", "daily_vol", 100),
    ("Sharpe Ratio", "daily_sharpe", 1),
    ("Sortino Ratio", "daily_sortino", 1),
    ("Max Drawdown", "max_drawdown", 100),
    ("Calmar Ratio", "calmar", 1),
]
labels = [r[0] for r in metric_rows]
agentic_vals = [stats.loc[key, "AgenticBL"] * scale for _, key, scale in metric_rows]
equal_vals = [stats.loc[key, "EqualWeight"] * scale for _, key, scale in metric_rows]

fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=equal_vals, name="EqualWeight", marker_color=COLOUR_EQUAL))
fig.add_trace(go.Bar(x=labels, y=agentic_vals, name="AgenticBL", marker_color=COLOUR_AGENTIC))
fig.update_layout(
    title="Performance Metrics",
    barmode="group", template="plotly_dark", height=420,
)
fig.show()

pd.DataFrame({"AgenticBL": agentic_vals, "EqualWeight": equal_vals}, index=labels).round(3)

,AgenticBL,EqualWeight
Total Return,134.030,122.443
Ann. Return (CAGR),18.551,17.352
Volatility (ann.),18.272,18.121
Sharpe Ratio,1.027,0.977
Sortino Ratio,1.510,1.429
Max Drawdown,-26.302,-28.306
Calmar Ratio,0.705,0.613
